# Starter Code — Stereo Vision dengan 1 HP

**Petunjuk persiapan foto, alat, dan hal yang harus dicatat ada di `README.md`.** Notebook ini untuk menjalankan kode dan mengisi hasil. Jalankan **A → G** berurutan.

## A — Konfigurasi dan pemeriksaan file
Isi tiga nilai yang diukur. Untuk Colab, ubah `UPLOAD_ZIP = True`.

In [ ]:
from pathlib import Path
import os, zipfile, csv
import cv2
import numpy as np
import matplotlib.pyplot as plt

# === ISI SESUAI README ===
PATTERN_SIZE = (9, 6)        # sudut dalam; papan cetak 10 x 7 kotak
SQUARE_SIZE_MM = 25.0        # GANTI dengan ukuran satu kotak hasil penggaris
BASELINE_CM = 10.0           # GANTI dengan perpindahan pusat kamera L -> R
UPLOAD_ZIP = False          # True hanya di Google Colab
DATA_DIR = Path('praktikum_stereo')

if UPLOAD_ZIP:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.zip'):
        raise ValueError('Unggah satu file ZIP folder praktikum_stereo.')
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        # Tolak entri yang berpotensi keluar dari direktori kerja
        for m in z.infolist():
            p = Path(m.filename)
            if p.is_absolute() or '..' in p.parts:
                raise ValueError('ZIP mengandung path tidak aman.')
        z.extractall('.')

CAL_DIR = DATA_DIR/'calibration'
STEREO_DIR = DATA_DIR/'stereo'
OUT = DATA_DIR/'output'
OUT.mkdir(parents=True, exist_ok=True)
image_ext = {'.jpg','.jpeg','.png','.bmp'}
cal_files = sorted(p for p in CAL_DIR.iterdir() if p.suffix.lower() in image_ext) if CAL_DIR.exists() else []
L_PATH, R_PATH = STEREO_DIR/'left.jpg', STEREO_DIR/'right.jpg'
print('OpenCV:',cv2.__version__)
print('Foto kalibrasi:',len(cal_files),'| kiri:',L_PATH.exists(),'| kanan:',R_PATH.exists())
print('Kotak:',SQUARE_SIZE_MM,'mm | baseline:',BASELINE_CM,'cm')
if SQUARE_SIZE_MM <= 0 or BASELINE_CM <= 0: raise ValueError('Ukuran kotak dan baseline harus positif.')
if len(cal_files) < 10: raise RuntimeError('Sediakan setidaknya 10 foto kalibrasi (disarankan 15–25).')
if not L_PATH.exists() or not R_PATH.exists(): raise FileNotFoundError('Periksa stereo/left.jpg dan stereo/right.jpg')

## B — Deteksi sudut chessboard dan ukur jarak dalam piksel
Hasil `contoh jarak 2 sudut` adalah nilai **foto**, bukan focal length. Foto gagal otomatis dilewati.

In [ ]:
cols, rows = PATTERN_SIZE
obj = np.zeros((cols*rows,3),np.float32)
obj[:,:2] = np.mgrid[0:cols,0:rows].T.reshape(-1,2) * SQUARE_SIZE_MM
objpoints, imgpoints, ok_files, bad_files, spacings = [],[],[],[],[]
size_wh = None
preview = None
for path in cal_files:
    img = cv2.imread(str(path))
    if img is None: bad_files.append(path.name); continue
    gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    wh = (gray.shape[1],gray.shape[0])
    if size_wh is None: size_wh=wh
    if wh != size_wh:
        bad_files.append(path.name+' (resolusi berbeda)'); continue
    found,corners = cv2.findChessboardCornersSB(gray, PATTERN_SIZE)
    if not found: bad_files.append(path.name); continue
    corners = corners.astype(np.float32)
    objpoints.append(obj.copy()); imgpoints.append(corners)
    ok_files.append(path.name)
    spacings.append(float(np.linalg.norm(corners[1,0]-corners[0,0])))
    if preview is None:
        preview=img.copy()
        cv2.drawChessboardCorners(preview,PATTERN_SIZE,corners,True)
print('Foto valid:',len(ok_files),'| Gagal:',len(bad_files))
print('Foto gagal:',bad_files)
print('Contoh jarak dua sudut dalam piksel:', round(spacings[0],2) if spacings else None)
if preview is not None:
    plt.figure(figsize=(10,6));plt.imshow(cv2.cvtColor(preview,cv2.COLOR_BGR2RGB));plt.title('Titik sudut yang terdeteksi');plt.axis('off');plt.show()
    cv2.imwrite(str(OUT/'corners_preview.jpg'),preview)
if len(ok_files) < 10: raise RuntimeError('Titik sudut berhasil dideteksi pada <10 foto. Ulangi/ambil foto baru.')

## C — Kalibrasi kamera: hasil `fx`, `fy`, `cx`, `cy`
Periksa RMS dan simpan hasilnya. **Tidak ada angka focal length asumsi**.

In [ ]:
rms,K,dist,rvecs,tvecs=cv2.calibrateCamera(objpoints,imgpoints,size_wh,None,None)
fx,fy,cx,cy = K[0,0],K[1,1],K[0,2],K[1,2]
print('Resolusi kalibrasi:',size_wh)
print('K =\n',K)
print(f'fx={fx:.2f} px | fy={fy:.2f} px | cx={cx:.2f} px | cy={cy:.2f} px')
print(f'RMS reprojection error={rms:.3f} px')
print('Distorsi:',dist.ravel())
np.savez(OUT/'calibration.npz',K=K,dist=dist,image_wh=size_wh,rms=rms,square_size_mm=SQUARE_SIZE_MM)
if not np.isfinite(rms) or rms > 2.0:
    print('PERHATIAN: RMS cukup besar. Periksa foto kalibrasi dan kesesuaian pola.')

## D — Baca stereo, koreksi distorsi, dan cek alignment
Jika foto kiri/kanan tidak cukup sejajar, **kode berhenti**. Ini bukan stereo rectification penuh. Gunakan objek diam dan rel yang lurus.

In [ ]:
left=cv2.imread(str(L_PATH));right=cv2.imread(str(R_PATH))
if left is None or right is None: raise ValueError('Gagal membaca foto stereo')
if left.shape!=right.shape or (left.shape[1],left.shape[0])!=size_wh:
    raise ValueError('Resolusi stereo harus sama dengan resolusi kalibrasi dan kedua gambar identik.')
# Undistort mempertahankan koordinat intrinsik K (P=K)
map1,map2=cv2.initUndistortRectifyMap(K,dist,None,K,size_wh,cv2.CV_32FC1)
left_u=cv2.remap(left,map1,map2,cv2.INTER_LINEAR)
right_u=cv2.remap(right,map1,map2,cv2.INTER_LINEAR)
lg=cv2.cvtColor(left_u,cv2.COLOR_BGR2GRAY)
rg=cv2.cvtColor(right_u,cv2.COLOR_BGR2GRAY)
orb=cv2.ORB_create(nfeatures=4500)
k1,des1=orb.detectAndCompute(lg,None)
k2,des2=orb.detectAndCompute(rg,None)
if des1 is None or des2 is None: raise RuntimeError('Fitur foto kurang. Gunakan objek/latar bertekstur.')
matches=cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(des1,des2,k=2)
good=[a for a,b in matches if a.distance < 0.75*b.distance]
if len(good) < 20: raise RuntimeError('Terlalu sedikit feature match. Ulangi foto dengan lebih banyak tekstur.')
pL=np.float32([k1[m.queryIdx].pt for m in good]);pR=np.float32([k2[m.trainIdx].pt for m in good])
# Fundamental matrix hanya untuk filter outlier, bukan mengganti rectification
F,mask=cv2.findFundamentalMat(pL,pR,cv2.FM_RANSAC,1.5,0.99)
if mask is None: raise RuntimeError('RANSAC gagal: pasangan foto sulit dicocokkan.')
inlier=mask.ravel().astype(bool)
if inlier.sum()<15: raise RuntimeError('Inlier kurang dari 15. Ulangi foto.')
dy=np.abs(pL[inlier,1]-pR[inlier,1])
dx=pL[inlier,0]-pR[inlier,0]
median_dy=float(np.median(dy));positive_fraction=float(np.mean(dx>0))
print('Matching:',len(good),'| Inlier:',int(inlier.sum()))
print(f'Median beda baris = {median_dy:.2f} px | Fraksi disparity positif = {positive_fraction:.1%}')
vis=cv2.drawMatches(left_u,k1,right_u,k2,[m for m,v in zip(good,inlier) if v][:40],None,flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(13,5));plt.imshow(cv2.cvtColor(vis,cv2.COLOR_BGR2RGB));plt.title('Contoh feature match inlier');plt.axis('off');plt.show()
cv2.imwrite(str(OUT/'matches.jpg'),vis)
ALIGNMENT_OK = (median_dy<=2.0 and positive_fraction>=0.70)
if not ALIGNMENT_OK:
    raise RuntimeError('Alignment belum cukup baik. Ulangi foto stereo tanpa rotasi/naik-turun; jangan pakai hasil depth.')
print('Alignment lolos pemeriksaan awal (bukan jaminan rectification sempurna).')

## E — Hitung disparity dan depth map
Untuk melihat hasil, objek dekat biasanya memiliki disparity lebih besar. Nilai tak valid dibiarkan sebagai `NaN`.

In [ ]:
if not ALIGNMENT_OK: raise RuntimeError('Alignment belum lolos.')
# StereoSGBM: numDisparities harus kelipatan 16
SGBM_NUM_DISPARITIES = 128
SGBM_BLOCK_SIZE = 7
sgbm=cv2.StereoSGBM_create(
    minDisparity=0, numDisparities=SGBM_NUM_DISPARITIES,
    blockSize=SGBM_BLOCK_SIZE,
    P1=8*1*SGBM_BLOCK_SIZE**2, P2=32*1*SGBM_BLOCK_SIZE**2,
    uniquenessRatio=10, speckleWindowSize=70, speckleRange=2,
    disp12MaxDiff=1, mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY)
raw=sgbm.compute(lg,rg).astype(np.float32)
disp=raw/16.0
valid=np.isfinite(disp)&(disp>0.5)
Z=np.full(disp.shape,np.nan,np.float32)
Z[valid]=(float(fx)*(BASELINE_CM/100.0))/disp[valid]
# Batasi nilai untuk visualisasi saja, bukan data mentah
print(f'Piksel disparity valid: {valid.mean()*100:.1f}%')
fig,ax=plt.subplots(1,3,figsize=(16,5))
ax[0].imshow(cv2.cvtColor(left_u,cv2.COLOR_BGR2RGB));ax[0].set_title('Foto kiri (terkoreksi)')
a=ax[1].imshow(np.ma.masked_invalid(np.where(valid,disp,np.nan)),cmap='plasma');ax[1].set_title('Disparity (px)');fig.colorbar(a,ax=ax[1],fraction=.04)
b=ax[2].imshow(np.ma.masked_invalid(Z),cmap='viridis',vmin=0,vmax=3);ax[2].set_title('Depth (m), skala warna 0–3 m');fig.colorbar(b,ax=ax[2],fraction=.04)
for x in ax:x.axis('off')
plt.tight_layout();plt.show()
np.save(OUT/'disparity_px.npy',np.where(valid,disp,np.nan))
np.save(OUT/'depth_m.npy',Z)
cv2.imwrite(str(OUT/'left_undistorted.jpg'),left_u)

## F — Ukur satu objek pada ROI
Ganti `ROI_X`, `ROI_Y`, dan `Z_TRUE_M` berdasarkan foto serta meteran. Pilih daerah bertekstur **di dalam** objek.

In [ ]:
# === ISI HASIL PENGAMATAN ===
ROI_X = 320            # GANTI: posisi horizontal target pada gambar kiri
ROI_Y = 240            # GANTI: posisi vertikal target
ROI_RADIUS = 10        # jendela 21 x 21 piksel
Z_TRUE_M = 1.0         # GANTI: jarak sebenarnya dari meteran (meter)

h,w=Z.shape
if not (0<=ROI_X<w and 0<=ROI_Y<h):raise ValueError('ROI berada di luar gambar.')
x0,x1=max(0,ROI_X-ROI_RADIUS),min(w,ROI_X+ROI_RADIUS+1)
y0,y1=max(0,ROI_Y-ROI_RADIUS),min(h,ROI_Y+ROI_RADIUS+1)
roi_z=Z[y0:y1,x0:x1];roi_d=disp[y0:y1,x0:x1]
mask_roi=np.isfinite(roi_z)&(roi_z>0)
if mask_roi.sum()<10:raise RuntimeError('ROI memiliki <10 piksel depth valid. Pilih area lain.')
Z_EST=float(np.median(roi_z[mask_roi]))
D_MED=float(np.median(roi_d[mask_roi]))
if Z_TRUE_M<=0:raise ValueError('Jarak meteran harus positif.')
ERROR_PCT=abs(Z_EST-Z_TRUE_M)/Z_TRUE_M*100
print(f'disparity ROI={D_MED:.2f} px | Z estimasi={Z_EST:.3f} m | meteran={Z_TRUE_M:.3f} m | error={ERROR_PCT:.2f}%')
shown=cv2.cvtColor(left_u,cv2.COLOR_BGR2RGB).copy()
cv2.rectangle(shown,(x0,y0),(x1-1,y1-1),(255,0,0),3)
plt.figure(figsize=(10,6));plt.imshow(shown);plt.title(f'ROI | Depth={Z_EST:.3f} m | Error={ERROR_PCT:.1f}%');plt.axis('off');plt.show()

## G — Simpan satu baris percobaan
Jalankan setiap kali mengukur jarak baru. **Sebelum uji berikutnya:** ganti foto `left.jpg/right.jpg`, lalu ulangi **D → G**. Jangan mengulang C bila parameter kamera tetap.

In [ ]:
TEST_ID = 'uji_01'  # GANTI: uji_01, uji_02, uji_03
csv_path=OUT/'hasil_pengukuran.csv'
record={
    'uji':TEST_ID,'baseline_cm':BASELINE_CM,'square_size_mm':SQUARE_SIZE_MM,
    'fx_px':round(float(fx),3),'fy_px':round(float(fy),3),
    'rms_px':round(float(rms),3),'median_vertical_error_px':round(median_dy,3),
    'disparity_roi_px':round(D_MED,3),'depth_est_m':round(Z_EST,4),
    'depth_meter_m':round(Z_TRUE_M,4),'error_pct':round(ERROR_PCT,2)}
rows=[]
if csv_path.exists():
    with csv_path.open(encoding='utf-8',newline='') as fh:rows=list(csv.DictReader(fh))
rows=[r for r in rows if r['uji']!=TEST_ID]+[record]
with csv_path.open('w',encoding='utf-8',newline='') as fh:
    writer=csv.DictWriter(fh,fieldnames=list(record));writer.writeheader();writer.writerows(rows)
print('Tersimpan:',csv_path)
for r in rows:print(r['uji'],'| jarak meteran',r['depth_meter_m'],'m | estimasi',r['depth_est_m'],'m | error',r['error_pct'],'%')

## Tugas analisis dan flowchart
**Ikuti format pengumpulan di README bagian 8.** Buat sendiri dua flowchart: (1) **A→B→C** dengan keputusan berhasil/gagal mendeteksi sudut; (2) **D→E→F→G** dengan keputusan alignment/ROI valid. Bandingkan minimal tiga jarak dan tulis penyebab error.

**Batasan:** proses ini menggunakan pendekatan translasi horizontal murni. Pemeriksaan alignment saja **tidak membuktikan** seluruh asumsi geometri terpenuhi; jangan melaporkan hasil sebagai jarak metrik terverifikasi tanpa perbandingan meteran.